In [2]:
import torch
import requests
from PIL import Image
from torchvision import transforms
from transformers import XLMRobertaTokenizer
import torch.nn.functional as F
import pandas as pd
import os

# GPU 사용 설정 (사용할 GPU 번호 지정, 필요시 수정)
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

# BEiT-3 저장소의 필요한 클래스와 설정 함수를 직접 가져옵니다.
from modeling_finetune import BEiT3ForCaptioning
from modeling_utils import _get_large_config

# --- 1. 설정 및 모델, 토크나이저 로드 ---

# 사용자 환경에 맞게 경로를 수정해주세요.
BASE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge"
MODEL_DIR = os.path.join(BASE_DIR, "unilm/beit3/")
DATA_DIR = os.path.join(BASE_DIR, "data/data_2/")

TOKENIZER_PATH = os.path.join(MODEL_DIR, "beit3.spm")
MODEL_WEIGHTS_PATH = os.path.join(MODEL_DIR, "beit3_large_indomain_patch16_224.pth")
TEST_CSV_PATH = os.path.join(DATA_DIR, "test.csv")
IMAGE_DIR = os.path.join(DATA_DIR, "test_input_images")
OUTPUT_CSV_PATH = os.path.join(DATA_DIR, "zeroshot_multichoice_ABCD.csv")

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 토크나이저 로드
tokenizer = XLMRobertaTokenizer(TOKENIZER_PATH)
print("✅ Tokenizer 로드 완료")

# --- 수정된 부분: '1', '2', '3', '4' 토큰의 ID를 가져옵니다 ---
choice_tokens = ['A', 'B', 'C', 'D']
choice_token_ids = tokenizer.convert_tokens_to_ids(choice_tokens)
mask_token_id = tokenizer.mask_token_id
print(f"선택지 토큰 ID: {choice_token_ids}")
print(f"마스크 토큰 ID: {mask_token_id}")

# --- 모델 로드 ---
args = _get_large_config(img_size=224)
args.vocab_size = 64010
model = BEiT3ForCaptioning(args=args)
state_dict = torch.load(MODEL_WEIGHTS_PATH, map_location='cpu')["model"]
keys_to_delete = [key for key in state_dict if key.startswith('mim_head')]
for key in keys_to_delete:
    del state_dict[key]
model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()
print("✅ 모델 가중치 로드 및 초기화 완료")

# --- 2. 이미지 전처리 ---
transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
])

# --- 3. 제로샷 예측 함수 (로직 변경) ---
def predict_multiple_choice(image_path, text_prompt, model, tokenizer, transform, device):
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)
    except Exception as e:
        print(f"이미지를 로드할 수 없습니다: {image_path} - {e}")
        return None, None

    encoding = tokenizer(text_prompt, return_tensors="pt", max_length=512, truncation=True)
    input_ids = encoding.input_ids.to(device)
    attention_mask = encoding.attention_mask.to(device)

    mask_token_indices = torch.where(input_ids == mask_token_id)[1]
    if len(mask_token_indices) == 0:
        print(f"프롬프트에서 [MASK] 토큰을 찾을 수 없습니다: {text_prompt}")
        return None, None

    with torch.no_grad():
        outputs = model.beit3(
            textual_tokens=input_ids,
            visual_tokens=image_tensor,
            text_padding_position=attention_mask.bool().logical_not()
        )
        encoder_out = outputs["encoder_out"]
        batch_indices = torch.arange(encoder_out.size(0))
        masked_token_feats = encoder_out[batch_indices, mask_token_indices]
        mlm_logits = model.mlm_head(masked_token_feats)

        # '1', '2', '3', '4' 토큰에 대한 logits만 추출
        choice_logits = mlm_logits[:, choice_token_ids]
        
        # Softmax를 적용하여 확률 계산
        choice_probabilities = F.softmax(choice_logits, dim=-1).squeeze()
        
        # 가장 확률이 높은 선택지의 인덱스(0, 1, 2, 3)를 찾음
        best_choice_index = torch.argmax(choice_probabilities).item()
        
        return best_choice_index, choice_probabilities.cpu().numpy()

# --- 4. 메인 실행 로직 (로직 변경) ---

df = pd.read_csv(TEST_CSV_PATH)
results = []
option_letters = ['A', 'B', 'C', 'D']

for index, row in df.iterrows():
    test_id = row['ID']
    question = row['Question']
    image_filename = os.path.basename(row['img_path'])
    image_path = os.path.join(IMAGE_DIR, image_filename)

    # --- 수정된 부분: 하나의 통합 프롬프트를 생성합니다 ---
    prompt = f"{question} A: {row['A']} B: {row['B']} C: {row['C']} D: {row['D']}. The correct answer is {tokenizer.mask_token}"

    print(f"\n--- ID: {test_id} 처리 중 ---")
    print(f"이미지: {image_path}")
    print(f"프롬프트: {prompt[:150]}...") # 프롬프트 일부 출력

    # 통합 프롬프트로 한 번만 예측 실행
    best_idx, probs = predict_multiple_choice(image_path, prompt, model, tokenizer, transform, device)

    if best_idx is not None:
        # 예측된 인덱스(0,1,2,3)를 정답 문자(A,B,C,D)로 변환
        predicted_answer = option_letters[best_idx]
        print(f"  예측된 확률 (1,2,3,4): {[f'{p:.4f}' for p in probs]}")
        print(f"-> 최종 선택: {predicted_answer}")
    else:
        predicted_answer = '-' # 예측 실패 시

    results.append({'ID': test_id, 'answer': predicted_answer})

# --- CSV 파일 저장 ---
submission_df = pd.DataFrame(results)
submission_df.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"\n✅ 모든 작업 완료! 예측 결과가 '{OUTPUT_CSV_PATH}' 파일에 저장되었습니다.")

✅ Tokenizer 로드 완료
선택지 토큰 ID: [370, 890, 468, 652]
마스크 토큰 ID: 64001
✅ 모델 가중치 로드 및 초기화 완료

--- ID: TEST_000 처리 중 ---
이미지: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test_input_images/TEST_000.jpg
프롬프트: What types of fruits are visible in the image? A: Bananas and grapes placed in baskets B: Apples and oranges displayed on the counter C: Peaches and p...
  예측된 확률 (1,2,3,4): ['0.1746', '0.1203', '0.1404', '0.5647']
-> 최종 선택: D

--- ID: TEST_001 처리 중 ---
이미지: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test_input_images/TEST_001.jpg
프롬프트: What type of cuisine are these dishes most commonly associated with? A: Chinese, famous for dumplings and noodles. B: Mexican, featuring tacos and enc...
  예측된 확률 (1,2,3,4): ['0.2847', '0.2408', '0.2891', '0.1854']
-> 최종 선택: C

--- ID: TEST_002 처리 중 ---
이미지: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test_input_images/TEST_002.jpg
프롬프트: What is a common reason people doing this activity? A